In [11]:
! pip install langchain langchain-community langchain-openai langchain-huggingface langchain-text-splitters langchain-chroma langchain-classic pypdf rank-bm25 python-dotenv
! pip install ragas datasets sentence-transformers

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [15]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_community.retrievers import BM25Retriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from datasets import Dataset

C:\Users\igors\AppData\Local\Temp\ipykernel_7136\746070053.py:16: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
C:\Users\igors\AppData\Local\Temp\ipykernel_7136\746070053.py:16: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
C:\Users\igors\AppData\Local\Temp\ipykernel_7136\746070053.py:16: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ra

In [17]:
load_dotenv()


True

In [28]:
test_queries = [
    "O que é lógica proposicional segundo a apostila?",
    "Como a apostila define uma proposição?",
    "O que são conectivos lógicos e quais são apresentados no material?",
    "O que é uma tabela-verdade e para que ela é utilizada?",
    "Como a apostila define tautologia, contradição e contingência?"
]

ground_truths = [
    "Lógica proposicional é o ramo da lógica que estuda proposições e as relações entre elas por meio de conectivos lógicos.",
    "Proposição é toda sentença declarativa que pode ser classificada como verdadeira ou falsa, mas não ambas.",
    "Conectivos lógicos são operadores que conectam proposições, como negação (¬), conjunção (∧), disjunção (∨), condicional (→) e bicondicional (↔).",
    "Tabela-verdade é um método utilizado para determinar o valor lógico de proposições compostas a partir dos valores lógicos das proposições simples.",
    "Tautologia é uma proposição composta que é sempre verdadeira; contradição é sempre falsa; contingência é aquela que pode ser verdadeira ou falsa dependendo dos valores das proposições componentes."
]

In [18]:
model = ChatOpenAI(
    model="llama3.3-70b-instruct",
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base="https://inference.do-ai.run/v1",
    temperature=0.7,
    max_tokens=1024,
)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

C:\Users\igors\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\igors\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6504.27it/s]
MPNetModel LOAD 

In [19]:
loader = PyPDFDirectoryLoader("../docs/")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=120,
    add_start_index=True,
)
all_splits = text_splitter.split_documents(docs)
print(f"Documentos divididos em {len(all_splits)} chunks.")

Documentos divididos em 155 chunks.


In [20]:
vector_store = Chroma(
    collection_name="hybrid_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",
)
vector_store.add_documents(documents=all_splits)
vector_retriever = vector_store.as_retriever(search_kwargs={"k": 5})

In [21]:
bm25_retriever = BM25Retriever.from_documents(all_splits, k=5)

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.4, 0.6],
)

In [23]:
prompt = ChatPromptTemplate.from_template(
    """Você é um assistente útil. Use o contexto abaixo para responder a pergunta.
Se não souber a resposta com base no contexto, diga que não sabe.

Contexto:
{context}

Pergunta: {question}

Resposta:"""
)

def format_docs(docs):
    return "".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": hybrid_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [29]:
eval_llm = ChatOpenAI(
    model="llama3.3-70b-instruct",
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base="https://inference.do-ai.run/v1",
    temperature=0,
)

print("Coletando respostas para avaliação RAGAS...")
ragas_data = []

for i, query in enumerate(test_queries):
    print(f"  [{i+1}/{len(test_queries)}] {query}")
    answer = rag_chain.invoke(query)
    context_docs = hybrid_retriever.invoke(query)
    contexts = [doc.page_content for doc in context_docs]
    ragas_data.append({
        "question": query,
        "answer": answer,
        "contexts": contexts,
        "ground_truth": ground_truths[i]
    })

dataset = Dataset.from_list(ragas_data)

print("Executando avaliação RAGAS...")
result = evaluate(
    dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=eval_llm,
    embeddings=embeddings,
)

print("=== RESULTADOS RAGAS ===")
print(result)

df = result.to_pandas()
print("Detalhes por query:")
print(df.to_string())

Coletando respostas para avaliação RAGAS...
  [1/5] O que é lógica proposicional segundo a apostila?
  [2/5] Como a apostila define uma proposição?
  [3/5] O que são conectivos lógicos e quais são apresentados no material?
  [4/5] O que é uma tabela-verdade e para que ela é utilizada?
  [5/5] Como a apostila define tautologia, contradição e contingência?
Executando avaliação RAGAS...


Evaluating:   5%|▌         | 1/20 [00:04<01:25,  4.49s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 20/20 [02:02<00:00,  6.12s/it]


=== RESULTADOS RAGAS ===
{'faithfulness': 0.9283, 'answer_relevancy': 0.8784, 'context_precision': 0.5852, 'context_recall': 0.8000}
Detalhes por query:
                                                           user_input                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          